In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
df_bronze=spark.read.format("csv").option("inferschema","true").option("header","true")\
    .load("/Volumes/aws_catalog/default/aws_databricks/project/gross_price/*.csv")

df_bronze=df_bronze.withColumn("read_timestamp", current_timestamp())

df_bronze=df_bronze.select("*","_metadata.file_name","_metadata.file_size")

display(df_bronze.limit(5))

In [0]:
df_bronze.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.bronze.price")

In [0]:
import pandas as pd
df_silver=spark.read.table("fmcg.bronze.price")
df_silver=df_silver.toPandas()
df_silver['month']=pd.to_datetime(df_silver['month'], format="mixed", errors="coerce")
df_silver=spark.createDataFrame(df_silver)
df_silver=df_silver.withColumn("month", to_date(col("month")))
display(df_silver)

In [0]:
df_silver.select("month").distinct().show()

In [0]:
df_silver=df_silver.withColumn("gross_price",regexp_replace(col("gross_price"),"-",""))\
    .withColumn("gross_price",regexp_replace(col("gross_price"),"unknown","0"))\
    .withColumn("gross_price",regexp_replace(col("gross_price"),"not_available","0"))\
    .withColumn("gross_price",col("gross_price").cast("double"))

In [0]:
df_products=spark.read.table("fmcg.silver.products")
df_joined=df_silver.join(df_products.select("product_id","product_code"), on="product_id",how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")
display(df_joined)

In [0]:
df_joined.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.silver.price")

In [0]:
df_gold=spark.read.table("fmcg.silver.price")
df_gold=df_gold.select("product_code","month","gross_price")
display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.gold.sb_dim_price")

In [0]:
df_gold_price=spark.read.table("fmcg.gold.sb_dim_price")
display(df_gold_price.limit(5))

In [0]:
df_gold_price=df_gold_price.withColumn("year",year(col("month")))\
    .withColumn("is_zero", when(col("gross_price")==0,1).otherwise(0))


In [0]:
w = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(col("is_zero"), col("month").desc())
)


df_gold_price = df_gold_price.withColumn("rnk", row_number().over(w))\
      .filter(col("rnk") == 1)

display(df_gold_price)

In [0]:
df_gold_price=df_gold_price.select("product_code", "year", "gross_price")\
    .withColumnRenamed("gross_price","price_inr")\
    .select("product_code","price_inr","year")


In [0]:
df_gold_price=df_gold_price.withColumn("year", col("year").cast("string"))
display(df_gold_price.limit(5))

In [0]:
df_gold_price.printSchema()

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")


delta_table.alias("target").merge(
    source=df_gold_price.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()